# 01 · Calidad de la extracción

Antes de mirar señales hay que saber **cuánto se puede confiar en los datos**. Este notebook mide:

1. Cobertura de textos (PDFs bajados vs. pendientes)
2. Calidad del parser por etapa y año
3. Resolución de CUIT y la lista priorizada de pendientes
4. Inconsistencias de montos
5. **Muestra aleatoria para revisión manual**: la única forma de conocer la precisión real

Solo lectura: no modifica la base.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from monitor_licitaciones.consultas import Base

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
db = Base()   # usa MLC_DSN; conexión de solo lectura
print(db.valor("SELECT current_database()"), "| parser", db.valor("SELECT max(parser_version) FROM bo.acto"))

## 1. Cobertura de textos
`pendiente` = normas de la sección Licitaciones cuyo PDF todavía no se bajó (`python -m monitor_licitaciones pdfs`).

In [ ]:
doc = db.df("SELECT anio, estado, documentos FROM analitica.calidad_documentos ORDER BY anio")
tabla = doc.pivot_table(index="anio", columns="estado", values="documentos", aggfunc="sum", fill_value=0)
tabla["% con texto"] = (100 * tabla.get("ok", 0) / tabla.sum(axis=1)).round(1)
display(tabla)
tabla.drop(columns="% con texto").plot.bar(stacked=True, figsize=(9, 3.5), title="Documentos por año y estado")
plt.ylabel("documentos"); plt.tight_layout(); plt.show()

## 2. Parser por etapa y año
Lo que importa para las señales son **adjudicación** y **preadjudicación**: código, monto y firma ganadora.

In [ ]:
cp = db.df("SELECT * FROM analitica.calidad_parser WHERE etapa IN ('adjudicacion','preadjudicacion') ORDER BY anio, etapa")
display(cp)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for i, col in enumerate(["pct_alta", "pct_con_monto", "pct_con_codigo"]):
    cp.pivot(index="anio", columns="etapa", values=col).plot(ax=ax[i], marker="o", title=col)
    ax[i].set_ylim(0, 100)
plt.tight_layout(); plt.show()

In [ ]:
# ¿Qué falta en los actos que no son de confianza alta?
db.df("""SELECT etapa, faltantes, count(*) AS actos FROM bo.acto
          WHERE etapa IN ('adjudicacion','preadjudicacion') AND confianza <> 'alta'
          GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 15""")

## 3. Resolución de CUIT
Orden de confianza: `cuit_en_texto` > `alias` > `padron_exacto` > `padron_base` > `padron_trgm`.
Lo `sin_resolver` incluye personas humanas (no están en el padrón de sociedades) y **homónimos**: el padrón tiene varias
sociedades con el mismo nombre y la regla no adivina.

In [ ]:
res = db.df("SELECT * FROM analitica.calidad_resolucion")
piv = res.pivot_table(index="rol", columns="resolucion", values="firmas", fill_value=0)
display(piv)
(piv.div(piv.sum(axis=1), axis=0) * 100).plot.barh(stacked=True, figsize=(10, 3.5), title="% de firmas por método de resolución")
plt.xlabel("%"); plt.legend(bbox_to_anchor=(1, 1)); plt.tight_layout(); plt.show()

### Pendientes priorizados
Los nombres sin CUIT con más actos. Si reconocés el candidato correcto, cargalo como alias **manual** (tiene prioridad sobre
todo lo automático) y volvé a correr `python -m monitor_licitaciones resolver`.

```sql
INSERT INTO ref.empresa_alias (nombre_norm, cuit, fuente) VALUES ('GBO ARGENTINA SA', 30708003391, 'manual');
```

In [ ]:
db.df("""SELECT p.*, ref.candidatos_cuit(p.ejemplo) AS candidatos
          FROM (SELECT * FROM analitica.pendientes_cuit ORDER BY actos DESC LIMIT 40) p
          ORDER BY p.actos DESC""")

## 4. Montos que no cierran
Actos donde la suma de montos por firma difiere más de 1% del total declarado. Pueden ser errores del propio Boletín o del
parser: conviene abrir el PDF (`url_norma`).

In [ ]:
cm = db.df("SELECT * FROM analitica.control_montos ORDER BY abs(pct_diferencia) DESC")
print(len(cm), "actos con diferencia > 1%")
cm.head(25)

## 5. Muestra para revisión manual
Genera un CSV con una muestra aleatoria **reproducible** de adjudicaciones y preadjudicaciones. Abrilo, compará cada fila
con el texto y completá las columnas `ok_*` con 1 (bien) o 0 (mal). Con 100 casos el error de la estimación ronda ±10 pp.

In [ ]:
N, SEMILLA = 100, "revision-1"   # misma semilla = misma muestra
muestra = db.df("""
  SELECT a.id_norma, a.etapa, a.codigo_proceso, a.monto_total, a.moneda, a.n_ofertas, a.unica_oferta,
         (SELECT string_agg(af.nombre_raw || ' [' || af.rol || coalesce(', ' || af.cuit, '') ||
                 coalesce(', $' || af.monto, '') || ']', ' | ' ORDER BY af.orden)
            FROM bo.acto_firma af WHERE af.id_norma = a.id_norma) AS firmas,
         n.url_norma, left(d.texto, 3000) AS texto,
         NULL::int AS ok_codigo, NULL::int AS ok_monto, NULL::int AS ok_ganadoras, NULL::int AS ok_sin_basura,
         NULL::text AS comentario
  FROM bo.acto a JOIN bo.norma n USING (id_norma) JOIN bo.documento d USING (id_norma)
  WHERE a.etapa IN ('adjudicacion','preadjudicacion')
  ORDER BY md5(a.id_norma::text || %s) LIMIT %s""", (SEMILLA, N))
ruta = os.path.join("..", "revision_manual.csv")
if not os.path.exists(ruta):
    muestra.to_csv(ruta, index=False, encoding="utf-8-sig")
    print("Muestra guardada en", os.path.abspath(ruta))
else:
    print("Ya existe", os.path.abspath(ruta), "— no se pisa (borralo para generar otra)")
muestra[["id_norma", "etapa", "codigo_proceso", "monto_total", "firmas"]].head(10)

In [ ]:
# Después de completar el CSV: precisión estimada con intervalo de confianza 95% (Wilson)
import math
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),) * 3
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; m = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return round(100*p, 1), round(100*(c-m), 1), round(100*(c+m), 1)

if os.path.exists(ruta):
    rev = pd.read_csv(ruta)
    cols = [c for c in rev.columns if c.startswith("ok_")]
    filas = []
    for c in cols:
        v = rev[c].dropna()
        filas.append((c, len(v), *wilson(int(v.sum()), len(v))))
    display(pd.DataFrame(filas, columns=["campo", "revisados", "% correcto", "IC95 inf", "IC95 sup"]))